# ETL de los Festivos

Este *notebook* realiza *web scraping* sobre la página de Calendarios Ideal y extrae los festivos y sus respectivas fechas.

In [27]:
# Importar librerías
import os, requests
from bs4 import BeautifulSoup
from datetime import datetime, date, timedelta, time
import pandas as pd
import numpy as np

In [28]:
def scrape_holidays_crevillente(year):
    """
    Extrae festivos oficiales de Crevillente (Alicante) para un año específico mediante web scraping.
    Realiza scraping de la web de calendarios laborales, parsea fechas en español
    y excluye automáticamente festivos de la primera semana de octubre.
    
    Args:
       - year (int): Año del cual extraer los festivos
    
    Returns:
       - pd.DataFrame: DataFrame con columnas "fecha" (date) y "festividad" (str),
                    ordenado por fecha y sin duplicados
    """
    url = f"https://calendarios.ideal.es/laboral/comunidad-valenciana/alicante/crevillent/{year}"
    headers = {"User-Agent": "Mozilla/5.0"}

    # Realizar consulta
    try:
        res = requests.get(url, headers=headers)
        res.raise_for_status()
        
    except Exception as e:
        print(f"Error al acceder al año {year}: {e}")
        return pd.DataFrame()

    soup = BeautifulSoup(res.text, "html.parser")

    # Mapeo de nombres de meses en español a número de mes
    months_map = {
        "enero": 1, "febrero": 2, "marzo": 3, "abril": 4,
        "mayo": 5, "junio": 6, "julio": 7, "agosto": 8,
        "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12
    }

    holidays = []

    # Buscar todos los <h2> con clase "v-a-t" (cada uno contiene ·, fecha y descripción)
    for h2 in soup.find_all("h2", class_="v-a-t"):
        spans = h2.find_all("span")
        if len(spans) < 2:
            continue

        # Texto de la fecha, por ejemplo: "lunes, 1 de enero de 2024"
        date_text = spans[0].get_text(strip=True).lower()

        # Ignorar la parte del día de la semana
        if "," in date_text:
            date_parts = date_text.split(",", 1)[1].strip()  # "1 de enero de 2024"
        else:
            date_parts = date_text

        # Ahora date_parts tiene formato "DD de <mes> de YYYY"
        try:
            parts = date_parts.split(" de ")
            if len(parts) == 3:
                day = int(parts[0])
                month_name = parts[1]
                year = int(parts[2])
                month_num = months_map.get(month_name)
                if month_num  is None:
                    raise ValueError(f"Mes desconocido: {month_name}")

                date_obj = date(year, month_num , day)
                
            else:
                raise ValueError(f"Formato inesperado: {date_parts}")
                
        except Exception as e:
            print(f"No se pudo parsear '{date_parts}': {e}")
            continue

        # Texto de la descripción (segundo <span>)
        holiday = spans[1].get_text(strip=True).lower()

        holidays.append({
            "fecha": date_obj,
            "festividad": holiday
        })

    df_holidays = pd.DataFrame(holidays).sort_values("fecha").drop_duplicates(subset="fecha", keep="last").reset_index(drop=True)
    
    # Eliminar festivos de la primera semana de octubre (se incorporan manualmente)
    mask = (date(year, 10, 1) <= df_holidays["fecha"]) & (df_holidays["fecha"] <= date(year, 10, 7))
    df_holidays = df_holidays[~mask]
    
    return df_holidays

In [29]:
def dates_moros_cristianos(year):
    """
    Genera fechas de las festividades de Moros y Cristianos para un año específico.
    Calcula las fechas del primer fin de semana de octubre y asigna cada festividad
    a su día correspondiente (viernes a lunes).
    
    Args:
       - year (int): Año para el cual generar las fechas
    
    Returns:
       - pd.DataFrame: DataFrame con columnas "fecha" (date) y "festividad" (str)
                    para los 4 días de celebración
    """
    # Primer día de octubre
    first_oct = date(year, 10, 1)

    # Calcular primer sábado: weekday() -> L=0 ... D=6; sábado=5
    delta_to_sat = (5 - first_oct.weekday()) % 7 # Cuánto falta para sábado
    sat = first_oct + timedelta(days=delta_to_sat) # Pimer sábado de octubre

    # Derivar las demás fechas
    fri = sat - timedelta(days=1)         # Desfile infantil
    sun = sat + timedelta(days=1)         # Entrada Mora
    mon = sat + timedelta(days=2)         # Fin de la semana grande

    fri_p = sat - timedelta(days=8)       # Cabos
    sat_p = sat - timedelta(days=7)       # Barracas
    sun_p = sat - timedelta(days=6)       # Barracas

    fri_q = sat - timedelta(days=15)      # Pre-cabos
    sat_q = sat - timedelta(days=14)      # Cena comparsas
    sun_q = sat - timedelta(days=13)      # Cena comparsas

    return pd.DataFrame([
        {"fecha": fri, "festividad": "desfile_infantil"},
        {"fecha": sat, "festividad": "entrada_cristiana"},
        {"fecha": sun, "festividad": "entrada_mora"},
        {"fecha": mon, "festividad": "fin_semana_grande"},

        {"fecha": fri_p, "festividad": "cabos"},
        {"fecha": sat_p, "festividad": "barracas_sab"},
        {"fecha": sun_p, "festividad": "barracas_dom"},

        {"fecha": fri_q, "festividad": "pre_cabos"},
        {"fecha": sat_q, "festividad": "cenas_sab"},
        {"fecha": sun_q, "festividad": "cenas_dom"}
    ])

In [30]:
def normalise_df_text(df, rename_vars=True, rename_values=True):
    """
    Normaliza el texto de un DataFrame aplicando transformaciones estándar:
    minúsculas, snake_case (espacios -> "_") y sin acentos.

    Args:
        - df (pd.DataFrame): DataFrame a normalizar
        - rename_vars (bool): Si True, normaliza nombres de columnas
        - rename_values (bool): Si True, normaliza valores de texto (solo columnas object)

    Returns:
        - pd.DataFrame: DataFrame normalizado
    """
    # Usar translate
    accent_translation = str.maketrans({
        "á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u",
    })

    df_normalised = df.copy()

    # Valores (solo columnas tipo object)
    if rename_values:
        text_cols = df_normalised.select_dtypes(include=["object"]).columns
        df_normalised[text_cols] = (
            df_normalised[text_cols]
            .apply(lambda s: (
                s.astype("string")                      # manejar NaN
                 .str.strip()                           # eliminar espacios antes y después
                 .str.lower()                           # convertir a minúsculas
                 .str.translate(accent_translation)     # quitar acentos carácter a carácter
                 .str.replace(r"\s+", "_", regex=True)  # espacios en blanco -> "_"
            ))
        )

    # Nombres de columnas
    if rename_vars:
        df_normalised.columns = [
            str(c).strip().lower().translate(accent_translation).replace(" ", "_")
            for c in df_normalised.columns
        ]

    return df_normalised

In [31]:
# Scrapear festivos
df_holidays_2024 = scrape_holidays_crevillente(2024)
df_holidays_2025 = scrape_holidays_crevillente(2025)
df_holidays_crevillente = (
    pd.concat([df_holidays_2024, df_holidays_2025]).reset_index(drop=True)
    .pipe(normalise_df_text)
)
df_holidays_crevillente["fecha"] = pd.to_datetime(df_holidays_crevillente["fecha"])

In [32]:
df_holidays_crevillente.head()

,fecha,festividad
0,2024-01-01,año_nuevo
1,2024-01-06,epifania_del_señor
2,2024-03-19,dia_de_san_jose
3,2024-03-28,jueves_santo
4,2024-03-29,viernes_santo


La semana santa no viene señalizada en los festivos. Comprobando los registros de años anteriores, vemos que no se suele especificar la semana santa en la página web. Por lo tanto, se incluirá manualmente en el dataset entorno al jueves santo, dado que este sí aparece siempre.

In [33]:
# Incorporar semana santa entorno al jueves santo
jueves_santo = df_holidays_crevillente.loc[df_holidays_crevillente["festividad"] == "jueves_santo", "fecha"]
days_holy_week = [
    ("domingo_ramos",         -4),
    ("lunes_santo",           -3),
    ("martes_santo",          -2),
    ("miercoles_santo",       -1),
    ("viernes_santo",          1),
    ("sabado_santo",           2),
    ("domingo_resurreccion",   3),
]

# Construir filas como lista de diccionarios
rows = [
    {"fecha": fecha + timedelta(days=offset), "festividad": nombre}
    for fecha in jueves_santo
    for nombre, offset in days_holy_week
]

df_holy_week = pd.DataFrame(rows)

In [34]:
# Moros y cristianos
df_moros_cristianos_2024 = dates_moros_cristianos(2024)
df_moros_cristianos_2025 = dates_moros_cristianos(2025)
df_holidays_moros_cristianos = (
    pd.concat([df_moros_cristianos_2024, df_moros_cristianos_2025]).reset_index(drop=True)
    .pipe(normalise_df_text)
)
df_holidays_moros_cristianos["fecha"] = pd.to_datetime(df_holidays_moros_cristianos["fecha"])

In [35]:
df_holidays_moros_cristianos

,fecha,festividad
0,2024-10-04,desfile_infantil
1,2024-10-05,entrada_cristiana
2,2024-10-06,entrada_mora
3,2024-10-07,fin_semana_grande
4,2024-09-27,cabos
5,2024-09-28,barracas_sab
6,2024-09-29,barracas_dom
7,2024-09-20,pre_cabos
8,2024-09-21,cenas_sab
9,2024-09-22,cenas_dom


In [36]:
# Concatenar dataframes de forma cronológica
df_all_holidays = pd.concat([df_holidays_crevillente, df_holy_week, df_holidays_moros_cristianos]).sort_values("fecha").reset_index(drop=True)
df_all_holidays.head()

,fecha,festividad
0,2024-01-01,año_nuevo
1,2024-01-06,epifania_del_señor
2,2024-03-19,dia_de_san_jose
3,2024-03-24,domingo_ramos
4,2024-03-25,lunes_santo


In [37]:
# Convertir a tipos
df_all_holidays["fecha"] = pd.to_datetime(df_all_holidays["fecha"])
df_all_holidays["festividad"] = df_all_holidays["festividad"].astype("object")

In [38]:
# Filtrar fechas
mask = (pd.to_datetime("2024-01-01") <= df_all_holidays["fecha"]) & (df_all_holidays["fecha"] <= pd.to_datetime("2025-05-25"))
df_all_holidays = df_all_holidays[mask]
df_all_holidays = df_all_holidays.drop_duplicates().reset_index(drop=True)

In [39]:
project_root = os.getcwd()
while project_root != os.path.dirname(project_root) and not os.path.exists(os.path.join(project_root, "configs", "config.yaml")):
    project_root = os.path.dirname(project_root)
store_path = os.path.join(project_root, "data", "raw")

In [40]:
# Guardar df en csv
df_all_holidays.to_csv(os.path.join(store_path, "all_holidays.csv"), index=False)